# casmi26-jc-onp-build: obscure natural-product held-out set

Builds a competition-shaped test set from host-approved public libraries (GNPS, MassBank,
Enveda-180 Zenodo), keeping only natural products that **no model in the jc pipeline has seen**:
not in train (train_structs), not in MassSpecGym (base GLACIER/ICEBERG training data), not in
the GLACIER fine-tune sets (casmi26-jc-mix labels). NIST-derived records are dropped.
Output feeds casmi26-jc-onp-eval. Generated by kernels/make_onp_build.py.

In [ ]:
import subprocess, sys, os, time
T0 = time.time()
try:
    import rdkit
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'])
import rdkit; print('rdkit', rdkit.__version__)
os.makedirs('/kaggle/working/onp_src', exist_ok=True)
os.makedirs('/tmp/dl', exist_ok=True)

In [ ]:
open('/kaggle/working/onp_src/common.py', 'w').write('"""Shared helpers: scorer-identity keys, adduct normalisation, spectrum readers."""\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem, RDLogger\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\n\nRDLogger.DisableLog("rdApp.*")\n\n# --------------------------------------------------------------------------\n# Identity\n# --------------------------------------------------------------------------\n\n_uncharger = rdMolStandardize.Uncharger()\n_frag_chooser = rdMolStandardize.LargestFragmentChooser()\n\n\n_tautomer_canonicalizer = rdMolStandardize.TautomerEnumerator()\n\n\ndef scorer_key(smiles: str | None) -> str | None:\n    """The competition metric\'s identity (Kaggle metric \'casmi-mean-reciprocal-rank\' v14, as copied\n    into the jc-* notebooks): RemoveStereochemistry, then TautomerEnumerator().Canonicalize,\n    then InChIKey14. Returns None when the metric would treat the SMILES as a non-match.\n    """\n    if not isinstance(smiles, str) or not smiles:\n        return None\n    try:\n        mol = Chem.MolFromSmiles(smiles)\n        if mol is None:\n            return None\n        Chem.RemoveStereochemistry(mol)\n        key = Chem.MolToInchiKey(_tautomer_canonicalizer.Canonicalize(mol))\n        return key[:14] if key else None\n    except (ValueError, RuntimeError, TypeError):\n        return None\n\n\ndef keys_parallel(smiles, fn, processes=None) -> dict:\n    """{smiles: fn(smiles)} over unique SMILES using all cores (tautomer canonicalisation is slow)."""\n    import multiprocessing as mp\n    uniq = list(dict.fromkeys(s for s in smiles if isinstance(s, str)))\n    if len(uniq) < 2000:\n        return {s: fn(s) for s in uniq}\n    with mp.Pool(processes) as pool:\n        return dict(zip(uniq, pool.map(fn, uniq, chunksize=500)))\n\n\ndef parent_key(smiles: str | None) -> str | None:\n    """Stricter identity for leak checks: largest fragment, neutralised, canonical\n    tautomer, stereo removed. Two structures sharing this key count as "seen"\n    even if their scorer keys differ (salts, charge states, ~8% tautomer copies).\n    """\n    if not isinstance(smiles, str) or not smiles:\n        return None\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None:\n        return None\n    try:\n        mol = _frag_chooser.choose(mol)\n        mol = _uncharger.uncharge(mol)\n        mol = rdMolStandardize.CanonicalTautomer(mol)\n    except Exception:  # standardiser can choke on odd valences; fall back to raw\n        pass\n    Chem.RemoveStereochemistry(mol)\n    key = Chem.MolToInchiKey(mol)\n    return key[:14] if key else None\n\n\ndef is_clean_answer(smiles: str) -> bool:\n    """RDKit-valid, single fragment, net-neutral: the shape a truth answer should have."""\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\n    if mol is None or len(Chem.GetMolFrags(mol)) != 1:\n        return False\n    return Chem.GetFormalCharge(mol) == 0\n\n\n# --------------------------------------------------------------------------\n# Adducts\n# --------------------------------------------------------------------------\n\n_ADDUCT_ALIASES = {  # keys are the competition\'s test.parquet spellings\n    "[M+H]+": ["M+H", "[M+H]", "[M+H]+", "M+H+", "[M+H]1+"],\n    "[M+Na]+": ["M+Na", "[M+Na]", "[M+Na]+", "M+Na+", "[M+Na]1+"],\n    "[M+NH4]+": ["M+NH4", "[M+NH4]", "[M+NH4]+", "M+NH4+", "[M+NH4]1+"],\n    "[M+K]+": ["M+K", "[M+K]", "[M+K]+", "M+K+", "[M+K]1+"],\n    "[M-H2O+H]+": ["M+H-H2O", "[M+H-H2O]", "[M+H-H2O]+", "[M-H2O+H]+", "M-H2O+H", "[M+H-H2O]1+"],\n    "[M-2H2O+H]+": ["M+H-2H2O", "[M+H-2H2O]+", "[M-2H2O+H]+", "M-2H2O+H"],\n    "[M-H]-": ["M-H", "[M-H]", "[M-H]-", "M-H-", "[M-H]1-"],\n    "[M+Cl]-": ["M+Cl", "[M+Cl]", "[M+Cl]-", "M+Cl-", "[M+Cl]1-"],\n    "[M+CH2O2-H]-": ["M+FA-H", "[M+FA-H]-", "[M+HCOO]-", "M+HCOO", "[M+CH2O2-H]-", "[M+HCOOH-H]-",\n                     "M+CH2O2-H", "[M+CHO2]-", "M+CHO2"],\n    "[M-H2O-H]-": ["M-H2O-H", "[M-H2O-H]-", "[M-H-H2O]-", "M-H-H2O"],\n}\n_ALIAS_TO_ADDUCT = {a.upper().replace(" ", ""): k for k, v in _ADDUCT_ALIASES.items() for a in v}\n\nSTANDARD_ADDUCTS = tuple(_ADDUCT_ALIASES)\n# What base (MassSpecGym) GLACIER can score; everything else is the fine-tune\'s subset.\nBASE_GLACIER_ADDUCTS = ("[M+H]+", "[M+Na]+")\n\n\ndef normalize_adduct(raw) -> str | None:\n    if not isinstance(raw, str):\n        return None\n    return _ALIAS_TO_ADDUCT.get(raw.strip().upper().replace(" ", ""))\n\n\ndef adduct_group(adduct: str | None) -> str:\n    if adduct in BASE_GLACIER_ADDUCTS:\n        return "base_glacier"\n    if adduct is None:\n        return "unknown"\n    return "ft_only"\n\n\n# --------------------------------------------------------------------------\n# Spectrum readers -> one row per spectrum with a common schema\n# --------------------------------------------------------------------------\n\nSPEC_COLUMNS = [\n    "source", "record_id", "smiles", "inchikey", "adduct_raw", "ionization_mode",\n    "instrument_type", "precursor_mz", "collision_energy_raw", "library", "date",\n    "ms2_mzs", "ms2_intensities",\n]\n\n_KEY_MAP = {\n    "smiles": "smiles", "computed smiles": "smiles", "ch$smiles": "smiles",\n    "inchikey": "inchikey", "ch$iupac_inchikey": "inchikey", "ch$link: inchikey": "inchikey",\n    "precursortype": "adduct_raw", "precursor_type": "adduct_raw", "adduct": "adduct_raw",\n    "ms$focused_ion: precursor_type": "adduct_raw", "ionmode": "ionization_mode",\n    "ion_mode": "ionization_mode", "ac$mass_spectrometry: ion_mode": "ionization_mode",\n    "instrument_type": "instrument_type", "instrumenttype": "instrument_type",\n    "ac$instrument_type": "instrument_type", "source_instrument": "instrument_type",\n    "precursormz": "precursor_mz", "precursor_mz": "precursor_mz", "pepmass": "precursor_mz",\n    "ms$focused_ion: precursor_m/z": "precursor_mz",\n    "collision_energy": "collision_energy_raw", "collisionenergy": "collision_energy_raw",\n    "collision_energies": "collision_energy_raw", "instrument": "instrument_type",\n    "ac$mass_spectrometry: collision_energy": "collision_energy_raw",\n    "name": "_name", "db#": "record_id", "accession": "record_id", "spectrumid": "record_id",\n    "spectrum_id": "record_id", "scans": "_scans", "library": "library",\n    "organism": "library", "comment": "_comment", "date": "date",\n    "library_name": "library", "librarymembership": "library",\n    "mslevel": "_mslevel", "ms_level": "_mslevel", "libraryquality": "_quality",\n}\n\n\ndef _finish(rec: dict, peaks: list, source: str, idx: int) -> dict | None:\n    if not peaks:\n        return None\n    if str(rec.get("_mslevel", "2")).strip() not in ("2", ""):\n        return None\n    if str(rec.get("_quality", "1")).strip() in ("4", "10"):  # GNPS \'challenge\'/unverified tiers\n        return None\n    if not rec.get("adduct_raw") and rec.get("_name"):  # GNPS: \'Compound name M+H\'\n        rec["adduct_raw"] = str(rec["_name"]).split()[-1]\n    out = {c: rec.get(c) for c in SPEC_COLUMNS}\n    out["source"] = source\n    if not out["record_id"]:\n        out["record_id"] = str(rec.get("_scans") or idx)\n    arr = np.asarray(peaks, dtype=float)\n    out["ms2_mzs"] = arr[:, 0]\n    out["ms2_intensities"] = arr[:, 1]\n    try:\n        out["precursor_mz"] = float(str(out["precursor_mz"]).split()[0])\n    except (TypeError, ValueError):\n        out["precursor_mz"] = np.nan\n    return out\n\n\ndef _parse_kv(line: str, sep: str):\n    k, _, v = line.partition(sep)\n    return k.strip().lower(), v.strip()\n\n\ndef read_msp(path: Path, source: str) -> list[dict]:\n    rows, rec, peaks, in_peaks, idx = [], {}, [], False, 0\n    with open(path, encoding="utf-8", errors="replace") as fh:\n        for line in fh:\n            line = line.strip()\n            if not line:\n                r = _finish(rec, peaks, source, idx)\n                if r:\n                    rows.append(r)\n                rec, peaks, in_peaks, idx = {}, [], False, idx + 1\n                continue\n            if in_peaks or (line[0].isdigit() and ":" not in line):\n                parts = re.split(r"[\\s;,]+", line)\n                try:\n                    peaks.append((float(parts[0]), float(parts[1])))\n                    continue\n                except (ValueError, IndexError):\n                    pass\n            k, v = _parse_kv(line, ":")\n            if k == "num peaks":\n                in_peaks = True\n            elif k in _KEY_MAP:\n                rec.setdefault(_KEY_MAP[k], v)\n            elif k == "comments":  # MoNA packs SMILES / instrument in Comments\n                for m in re.finditer(r\'"([^"=]+)=([^"]*)"\', v):\n                    kk = m.group(1).strip().lower()\n                    if kk in _KEY_MAP:\n                        rec.setdefault(_KEY_MAP[kk], m.group(2).strip())\n    r = _finish(rec, peaks, source, idx)\n    if r:\n        rows.append(r)\n    return rows\n\n\ndef read_mgf(path: Path, source: str) -> list[dict]:\n    rows, rec, peaks, idx = [], {}, [], 0\n    with open(path, encoding="utf-8", errors="replace") as fh:\n        for line in fh:\n            line = line.strip()\n            if line == "BEGIN IONS":\n                rec, peaks = {}, []\n            elif line == "END IONS":\n                r = _finish(rec, peaks, source, idx)\n                if r:\n                    rows.append(r)\n                idx += 1\n            elif line and (line[0].isdigit()) and "=" not in line:\n                parts = line.split()\n                try:\n                    peaks.append((float(parts[0]), float(parts[1])))\n                except (ValueError, IndexError):\n                    pass\n            elif "=" in line:\n                k, v = _parse_kv(line, "=")\n                if k in _KEY_MAP:\n                    rec.setdefault(_KEY_MAP[k], v)\n    return rows\n\n\ndef normalize_instrument(raw) -> str | None:\n    """Collapse vendor strings onto test.parquet\'s vocabulary (\'timsTOF\' for every test row)."""\n    if not isinstance(raw, str):\n        return None\n    r = raw.lower()\n    if "tims" in r:\n        return "timsTOF"\n    if "orbitrap" in r or "q exactive" in r or "exploris" in r or "hcd" in r:\n        return "Orbitrap"\n    if "tof" in r:\n        return "QTOF"\n    return raw\n\n\ndef read_table(path: Path, source: str, skip_ik14: set | None = None) -> list[dict]:\n    """Parquet/CSV/JSON already in a tabular layout (e.g. Enveda-180\'s parquet)."""\n    if path.suffix == ".parquet" or path.name.endswith(".parquet.gz"):\n        import pyarrow.parquet as pq\n        df = pq.read_table(path).to_pandas()\n    elif path.suffix in (".csv", ".tsv"):\n        df = pd.read_csv(path, sep="\\t" if path.suffix == ".tsv" else ",")\n    else:\n        df = pd.DataFrame(json.loads(path.read_text()))\n    lower = {c.lower(): c for c in df.columns}\n    ren = {}\n    if "instrument" in lower:  # specific model name beats a generic type like \'Q-TOF\'\n        ren[lower["instrument"]] = "instrument_type"\n    for k, v in _KEY_MAP.items():\n        if k in lower and not v.startswith("_") and v not in ren.values():\n            ren[lower[k]] = v\n    for alt, tgt in (("ms2_mzs", "ms2_mzs"), ("mzs", "ms2_mzs"),\n                     ("ms2_normalized_intensities", "ms2_intensities"),\n                     ("intensities", "ms2_intensities"), ("ms2_intensities", "ms2_intensities")):\n        if alt in lower and tgt not in ren.values():\n            ren[lower[alt]] = tgt\n    df = df[list(ren)].rename(columns=ren)\n    if skip_ik14 and "inchikey" in df.columns:\n        df = df[~df["inchikey"].astype(str).str[:14].isin(skip_ik14)]\n    rows = []\n    for i, r in enumerate(df.to_dict("records")):\n        out = {c: r.get(c) for c in SPEC_COLUMNS}\n        out["source"] = source\n        out["record_id"] = out["record_id"] or str(i)\n        if out["ms2_mzs"] is None:\n            continue\n        out["ms2_mzs"] = np.asarray(out["ms2_mzs"], dtype=float)\n        out["ms2_intensities"] = np.asarray(out["ms2_intensities"], dtype=float)\n        try:\n            out["precursor_mz"] = float(str(out["precursor_mz"]).split()[0])\n        except (TypeError, ValueError):\n            out["precursor_mz"] = np.nan\n        rows.append(out)\n    return rows\n\n\ndef read_spectra(path: str, skip_ik14: set | None = None) -> pd.DataFrame:\n    """`path` may be `name=file` to label the source; otherwise the file stem is used."""\n    source, _, file = path.rpartition("=")\n    p = Path(file)\n    source = source or p.stem\n    suf = p.suffix.lower()\n    if suf == ".msp":\n        rows = read_msp(p, source)\n    elif suf == ".mgf":\n        rows = read_mgf(p, source)\n    else:\n        rows = read_table(p, source, skip_ik14)\n    df = pd.DataFrame(rows, columns=SPEC_COLUMNS)\n    df["instrument_type"] = df["instrument_type"].map(normalize_instrument)\n    return df\n\n\ndef load_key_set(path: str | None, smiles_cols=("smiles", "SMILES", "canonical_smiles"),\n                 key_cols=("scorer_key", "inchikey14", "inchikey", "InChIKey")) -> tuple[set, set]:\n    """Load a structure table and return (scorer keys, parent keys).\n\n    Uses SMILES when present (recomputes both keys, so the shipped-key mismatch in\n    D742042 can\'t leak through); otherwise falls back to the first 14 chars of a key\n    column, which only fills the scorer-key set.\n    """\n    if not path:\n        return set(), set()\n    p = Path(path).expanduser()\n    if p.suffix == ".parquet":\n        import pyarrow.parquet as pq\n        names = pq.ParquetFile(p).schema_arrow.names\n        want = [c for c in (*smiles_cols, *key_cols) if c in names]\n        df = pq.read_table(p, columns=want).to_pandas()\n    elif p.suffix == ".txt":\n        df = pd.DataFrame({"inchikey": p.read_text().split()})\n    else:\n        df = pd.read_csv(p, sep="\\t" if p.suffix == ".tsv" else ",")\n    scol = next((c for c in smiles_cols if c in df.columns), None)\n    kcol = next((c for c in key_cols if c in df.columns), None)\n    skeys, pkeys = set(), set()\n    if kcol:\n        skeys |= {str(k)[:14] for k in df[kcol].dropna()}\n    if scol:\n        uniq = df[scol].dropna().unique()\n        skeys |= {k for k in keys_parallel(uniq, scorer_key).values() if k}\n        pkeys |= {k for k in keys_parallel(uniq, parent_key).values() if k}\n    return skeys, pkeys\n')

In [ ]:
open('/kaggle/working/onp_src/build_obscure_np_set.py', 'w').write('"""Build an honest "obscure natural product" held-out test set.\n\nEvery molecule kept is absent from train.parquet under both the scorer identity\nand a stricter parent identity (salt/charge/tautomer/stereo-stripped), so no\nmodel trained on train.parquet (FPNets, GLACIER fine-tunes, model_views, fragnet,\nrankers) has seen it. Output mimics the competition test layout so the existing\nkernels can run on it unchanged.\n\nExample (Mac):\n    python bench/obscure_np/build_obscure_np_set.py \\\n        --train ~/casmi26/data/train.parquet \\\n        --train-structs ~/casmi26/data/train_structs.parquet \\\n        --spectra enveda180=~/casmi26/ext/enveda180.mgf \\\n                  massbank=~/casmi26/ext/MassBank_NISTformat.msp \\\n                  mona=~/casmi26/ext/MoNA-export-LC-MS-MS_Spectra.msp \\\n                  gnps=~/casmi26/ext/ALL_GNPS_NO_PROPOGATED.mgf \\\n        --coconut ~/casmi26/data/coconut.parquet \\\n        --candidate-db ~/casmi26/data/pubchem_coconut_keys.parquet \\\n        --popularity ~/casmi26/data/pubchem_popularity.parquet \\\n        --exclude-panel ~/casmi26/data/np_panel_keys.txt \\\n        --out ~/casmi26/bench/obscure_np_v1\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom common import (\n    STANDARD_ADDUCTS, adduct_group, is_clean_answer, keys_parallel, load_key_set,\n    normalize_adduct, parent_key, read_spectra, scorer_key,\n)\n\n# Library names that are NIST or NIST-derived; never let them into an eval set\n# that could later feed training (D744470, D745832).\nNIST_MARKERS = ("nist", "gnps-nist14", "nist14", "nist20", "nist23")\n\n\ndef np_likeness(smiles_list):\n    """RDKit NP-likeness (Ertl 2008); higher = more NP-like. None if contrib missing."""\n    try:\n        from rdkit import Chem\n        from rdkit.Contrib.NP_Score import npscorer\n    except ImportError:\n        return None\n    model = npscorer.readNPModel()\n    out = []\n    for s in smiles_list:\n        m = Chem.MolFromSmiles(s)\n        out.append(npscorer.scoreMol(m, model) if m is not None else np.nan)\n    return np.asarray(out)\n\n\ndef normalize_mode(raw, adduct):\n    if isinstance(raw, str):\n        r = raw.strip().lower()\n        if r.startswith("pos") or r == "p" or r == "+":\n            return "positive"\n        if r.startswith("neg") or r == "n" or r == "-":\n            return "negative"\n    if adduct:\n        return "positive" if adduct.endswith("+") else "negative"\n    return None\n\n\ndef ce_array(raw: str, nce: bool, precursor_mz: float) -> np.ndarray:\n    """All energies in a CE string as an eV array, like test.parquet (\'20,40,60\' -> [20, 40, 60]).\n\n    NCE uses the usual Thermo approximation eV = NCE * m/z / 500. Unparseable -> empty array,\n    which the kernels already treat as "no CE".\n    """\n    vals = [float(x) for x in re.findall(r"\\d+(?:\\.\\d+)?", raw or "")]\n    if nce and np.isfinite(precursor_mz):\n        vals = [v * precursor_mz / 500.0 for v in vals]\n    return np.asarray(vals, dtype=float)\n\n\ndef resample(df: pd.DataFrame, per_mol: float, seed: int) -> pd.DataFrame:\n    """Draw spectra per molecule so the mean count matches the test (~3.03, D742055).\n\n    Counts are Poisson(per_mol) clipped to [1, available], which keeps singletons\n    present the way the hidden test has them, instead of a flat top-k cut.\n    """\n    rng = np.random.default_rng(seed)\n    parts = []\n    for _, g in df.groupby("molecule_id", sort=False):\n        k = int(np.clip(rng.poisson(per_mol), 1, len(g)))\n        parts.append(g.iloc[rng.choice(len(g), size=k, replace=False)])\n    return pd.concat(parts, ignore_index=True)\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--train", required=True, help="train.parquet (needs a SMILES or inchikey column)")\n    ap.add_argument("--train-structs", help="train_structs.parquet, merged into the \'seen\' set")\n    ap.add_argument("--extra-seen", nargs="*", default=[],\n                    help="More structure tables any pipeline model trained on (e.g. MassSpecGym, "\n                         "fine-tune sets). Their molecules are excluded too.")\n    ap.add_argument("--spectra", nargs="+", required=True, help="name=path (.msp/.mgf/.parquet/.csv)")\n    ap.add_argument("--coconut", help="COCONUT structures; membership counts as NP")\n    ap.add_argument("--np-score-min", type=float, default=0.3,\n                    help="NP-likeness cutoff for molecules not in --coconut (Ertl score)")\n    ap.add_argument("--candidate-db", help="Structure keys of the retrieval candidate pool "\n                                           "(PubChem ∪ COCONUT). Used to label class 2 vs 3.")\n    ap.add_argument("--popularity", help="Table with smiles/inchikey + a \'popularity\' column")\n    ap.add_argument("--obscure-max-pop", type=float, default=None,\n                    help="Popularity at or below this is \'obscure\' (default: bottom half)")\n    ap.add_argument("--exclude-panel", help="Keys to drop (e.g. the Enveda NP panel) so sets stay disjoint")\n    ap.add_argument("--adducts", nargs="*", default=list(STANDARD_ADDUCTS))\n    ap.add_argument("--instruments", nargs="*", default=None,\n                    help="Keep only these instrument_type values (substring, case-insensitive), e.g. timsTOF")\n    ap.add_argument("--min-peaks", type=int, default=3)\n    ap.add_argument("--max-precursor", type=float, default=1500.0)\n    ap.add_argument("--spectra-per-mol", type=float, default=3.03)\n    ap.add_argument("--draws", type=int, default=5, help="Independent resampling draws")\n    ap.add_argument("--max-molecules", type=int, default=None, help="Cap after filtering (random)")\n    ap.add_argument("--seed", type=int, default=26)\n    ap.add_argument("--out", required=True)\n    args = ap.parse_args()\n\n    out = Path(args.out).expanduser()\n    out.mkdir(parents=True, exist_ok=True)\n    log = {"args": vars(args), "funnel": {}}\n\n    def step(name, n):\n        log["funnel"][name] = int(n)\n        print(f"{name:<38} {n:>9,}")\n\n    # ---- seen structures --------------------------------------------------\n    seen_s, seen_p = load_key_set(args.train)\n    for extra in [args.train_structs, *args.extra_seen]:\n        s, p = load_key_set(extra)\n        seen_s |= s\n        seen_p |= p\n    step("seen scorer keys", len(seen_s))\n    step("seen parent keys", len(seen_p))\n    if not seen_p:\n        print("WARNING: no SMILES in seen tables; parent-key (tautomer/salt) leak check is OFF")\n\n    # ---- spectra ------------------------------------------------------------\n    # Rows whose stored InChIKey14 is already seen are skipped while reading (memory); the\n    # SMILES-based scorer/parent checks below still catch the ~1.6% whose stored key differs.\n    spec = pd.concat([read_spectra(p, skip_ik14=seen_s) for p in args.spectra], ignore_index=True)\n    step("spectra read", len(spec))\n\n    lib = spec["library"].fillna("").str.lower() + " " + spec["source"].str.lower()\n    spec = spec[~lib.str.contains("|".join(NIST_MARKERS))]\n    step("after dropping NIST-derived", len(spec))\n\n    spec["adduct"] = spec["adduct_raw"].map(normalize_adduct)\n    spec = spec[spec["adduct"].isin(args.adducts)]\n    step("after standard-adduct filter", len(spec))\n\n    if args.instruments:\n        inst = spec["instrument_type"].fillna("").str.lower()\n        spec = spec[inst.apply(lambda x: any(i.lower() in x for i in args.instruments))]\n        step("after instrument filter", len(spec))\n\n    npk = spec["ms2_mzs"].map(len)\n    spec = spec[(npk >= args.min_peaks) & (spec["precursor_mz"] <= args.max_precursor)]\n    step("after peak/precursor filter", len(spec))\n\n    # Structure identity, computed once per unique SMILES.\n    uniq = spec["smiles"].dropna().unique()\n    skey = keys_parallel(uniq, scorer_key)\n    pkey = keys_parallel(uniq, parent_key)\n    clean = {s: is_clean_answer(s) for s in uniq}\n    spec["scorer_key"] = spec["smiles"].map(skey)\n    spec["parent_key"] = spec["smiles"].map(pkey)\n    spec = spec[spec["scorer_key"].notna() & spec["smiles"].map(clean).fillna(False)]\n    step("after valid neutral single-fragment", len(spec))\n\n    unseen = ~spec["scorer_key"].isin(seen_s) & ~spec["parent_key"].isin(seen_p)\n    spec = spec[unseen]\n    step("after unseen-in-train filter", len(spec))\n\n    if args.exclude_panel:\n        ex_s, ex_p = load_key_set(args.exclude_panel)\n        spec = spec[~spec["scorer_key"].isin(ex_s) & ~spec["parent_key"].isin(ex_p)]\n        step("after excluding panel", len(spec))\n\n    # ---- one canonical structure per scorer key ----------------------------\n    mols = (spec.groupby("scorer_key")\n                .agg(smiles=("smiles", "first"), n_spectra=("record_id", "size"),\n                     sources=("source", lambda x: ",".join(sorted(set(x)))),\n                     adducts=("adduct", lambda x: ",".join(sorted(set(x)))))\n                .reset_index())\n    step("unique unseen molecules", len(mols))\n\n    # ---- NP filter ----------------------------------------------------------\n    coco_s, coco_p = load_key_set(args.coconut)\n    mols["in_coconut"] = mols["scorer_key"].isin(coco_s) | mols["smiles"].map(parent_key).isin(coco_p)\n    nps = np_likeness(mols["smiles"])\n    mols["np_score"] = nps if nps is not None else np.nan\n    if nps is None and not coco_s:\n        raise SystemExit("Need --coconut or RDKit Contrib NP_Score to decide what counts as an NP")\n    mols["is_np"] = mols["in_coconut"] | (mols["np_score"] >= args.np_score_min)\n    mols = mols[mols["is_np"]].copy()\n    step("unseen natural products", len(mols))\n\n    # ---- class + obscurity labels -----------------------------------------\n    if args.candidate_db:\n        cand_s, _ = load_key_set(args.candidate_db)\n        mols["cls"] = np.where(mols["scorer_key"].isin(cand_s), 2, 3)\n    else:\n        mols["cls"] = np.nan\n    if args.popularity:\n        pop = pd.read_parquet(args.popularity) if args.popularity.endswith(".parquet") else pd.read_csv(args.popularity)\n        if "scorer_key" not in pop.columns:\n            if "smiles" in pop.columns:\n                pop["scorer_key"] = pop["smiles"].map(scorer_key)\n            else:\n                kcol = next(c for c in pop.columns if "inchikey" in c.lower())\n                pop["scorer_key"] = pop[kcol].astype(str).str[:14]\n        pop = pop.groupby("scorer_key")["popularity"].max()\n        mols["popularity"] = mols["scorer_key"].map(pop).fillna(0.0)\n        thr = args.obscure_max_pop if args.obscure_max_pop is not None else mols["popularity"].median()\n        mols["obscure"] = mols["popularity"] <= thr\n    else:\n        mols["popularity"] = np.nan\n        mols["obscure"] = np.nan\n\n    if args.max_molecules and len(mols) > args.max_molecules:\n        mols = mols.sample(args.max_molecules, random_state=args.seed)\n        step("after --max-molecules cap", len(mols))\n\n    mols = mols.sort_values("scorer_key").reset_index(drop=True)\n    # Same id shapes as test.parquet (m_xxxxxx / s_xxxxxxxx); stable across rebuilds.\n    mols["molecule_id"] = ["m_" + hashlib.md5(k.encode()).hexdigest()[:8] for k in mols["scorer_key"]]\n    assert mols["molecule_id"].is_unique\n    spec = spec.merge(mols[["scorer_key", "molecule_id"]], on="scorer_key")\n\n    # ---- competition-shaped output -----------------------------------------\n    spec["ionization_mode"] = [normalize_mode(m, a) for m, a in zip(spec["ionization_mode"], spec["adduct"])]\n    spec["ms2_normalized_intensities"] = spec["ms2_intensities"].map(\n        lambda a: a / a.max() if len(a) and a.max() > 0 else a)\n    spec["base_peak_intensity"] = spec["ms2_intensities"].map(lambda a: float(a.max()) if len(a) else np.nan)\n    spec["collision_energy_orig"] = spec["collision_energy_raw"].astype(str)\n    is_nce = spec["collision_energy_orig"].str.contains("%|nce|hcd", case=False)\n    spec["collision_energy_orig_units"] = "eV"  # test.parquet is all eV; NCE is converted below\n    spec["collision_energy_ev"] = [\n        ce_array(raw, nce, pmz) for raw, nce, pmz in\n        zip(spec["collision_energy_orig"], is_nce, spec["precursor_mz"])]\n    test_cols = ["molecule_id", "spectrum_id", "ms2_mzs", "ms2_normalized_intensities",\n                 "base_peak_intensity", "adduct", "ionization_mode", "instrument_type",\n                 "precursor_mz", "collision_energy_ev", "collision_energy_orig",\n                 "collision_energy_orig_units"]\n\n    full = spec.copy()\n    full["spectrum_id"] = ["s_" + hashlib.md5(f"{a}:{b}:{i}".encode()).hexdigest()[:8]\n                           for i, (a, b) in enumerate(zip(full["source"], full["record_id"]))]\n    full["record_ref"] = full["source"] + ":" + full["record_id"].astype(str)\n    assert full["spectrum_id"].is_unique\n    full[test_cols + ["source", "record_ref", "scorer_key"]].to_parquet(out / "spectra_all.parquet", index=False)\n\n    for d in range(args.draws):\n        draw = resample(full, args.spectra_per_mol, args.seed + d)\n        draw[test_cols].to_parquet(out / f"test_draw{d}.parquet", index=False)\n    step("spectra in pool", len(full))\n\n    # Per-molecule truth + slice labels. adduct_group uses the full pool so a molecule\n    # is \'ft_only\' only if no base-GLACIER adduct exists anywhere for it.\n    has_base = full.groupby("molecule_id")["adduct"].agg(lambda x: any(adduct_group(a) == "base_glacier" for a in x))\n    mols["adduct_group"] = mols["molecule_id"].map(has_base).map({True: "base_glacier", False: "ft_only"})\n    tims = full.groupby("molecule_id")["instrument_type"].agg(\n        lambda x: any("tims" in str(v).lower() for v in x))\n    mols["timstof"] = mols["molecule_id"].map(tims)\n    mols.to_csv(out / "solution.csv", index=False)\n    pd.DataFrame({"molecule_id": mols["molecule_id"], "smiles": ";".join(["CCO"] * 25)}).to_csv(\n        out / "sample_submission.csv", index=False)\n\n    # Summary\n    summ = {\n        "molecules": len(mols),\n        "spectra_pool": len(full),\n        "by_source": full.groupby("source")["molecule_id"].nunique().to_dict(),\n        "by_adduct_group": mols["adduct_group"].value_counts().to_dict(),\n        "by_class": mols["cls"].value_counts(dropna=False).to_dict() if args.candidate_db else None,\n        "obscure_share": float(mols["obscure"].mean()) if args.popularity else None,\n        "by_adduct_spectra": full["adduct"].value_counts().to_dict(),\n    }\n    log["summary"] = summ\n    (out / "build_log.json").write_text(json.dumps(log, indent=2, default=str))\n    print(json.dumps(summ, indent=2, default=str))\n\n\nif __name__ == "__main__":\n    main()\n')

In [ ]:
open('/kaggle/working/onp_src/mrr_eval.py', 'w').write('"""MRR@25 on the obscure-NP set, sliced, with paired bootstrap against a baseline.\n\nUsage:\n    python bench/obscure_np/mrr_eval.py --solution OUT/solution.csv \\\n        --sub jc-c2=preds/jc-c2_draw0.csv --sub cand=preds/cand_draw0.csv \\\n        --baseline jc-c2\n\nPredictions may be wide (molecule_id + smiles_1..smiles_25, or any columns ending\nin a rank number) or long (molecule_id, rank, smiles). Candidates are deduplicated\non the scorer key after ranking, as the metric does, so duplicate tails cost a slot\nonly in the way the grader would count them.\n\nRule from the handoff: nothing ships unless it beats jc-c2 here. A "win" below\nmeans the bootstrap 90% interval of the paired difference is above zero.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport re\n\nimport numpy as np\nimport pandas as pd\n\nfrom common import adduct_group, scorer_key\n\nK = 25\n\n\ndef load_preds(path: str) -> dict[str, list[str]]:\n    df = pd.read_csv(path) if not path.endswith(".parquet") else pd.read_parquet(path)\n    if set(df.columns) == {"molecule_id", "smiles"}:  # competition format: \';\'-joined top 25\n        return {r.molecule_id: [x.strip() for x in str(r.smiles).split(";") if x.strip()]\n                for r in df.itertuples()}\n    if "rank" in df.columns:\n        smi = next(c for c in df.columns if c.lower() in ("smiles", "pred", "candidate"))\n        df = df.sort_values(["molecule_id", "rank"])\n        return df.groupby("molecule_id")[smi].apply(list).to_dict()\n    rank_cols = sorted((c for c in df.columns if re.search(r"\\d+$", c) and c != "molecule_id"),\n                       key=lambda c: int(re.search(r"(\\d+)$", c).group(1)))\n    return {r["molecule_id"]: [r[c] for c in rank_cols if isinstance(r[c], str)]\n            for r in df.to_dict("records")}\n\n\ndef reciprocal_ranks(preds: dict[str, list[str]], truth: pd.Series, cache: dict) -> pd.Series:\n    out = {}\n    for mid, tkey in truth.items():\n        seen, rank = set(), 0\n        rr = 0.0\n        for s in preds.get(mid, []):\n            k = cache.get(s)\n            if k is None and s not in cache:\n                k = cache[s] = scorer_key(s)\n            if k is None or k in seen:\n                continue\n            seen.add(k)\n            rank += 1\n            if rank > K:\n                break\n            if k == tkey:\n                rr = 1.0 / rank\n                break\n        out[mid] = rr\n    return pd.Series(out)\n\n\ndef bootstrap_diff(a: pd.Series, b: pd.Series, n=2000, seed=0):\n    d = (a - b).to_numpy()\n    rng = np.random.default_rng(seed)\n    means = d[rng.integers(0, len(d), size=(n, len(d)))].mean(axis=1)\n    return d.mean(), np.percentile(means, 5), np.percentile(means, 95)\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--solution", required=True)\n    ap.add_argument("--sub", nargs="+", required=True, help="name=path")\n    ap.add_argument("--baseline", help="name of the --sub to compare others against")\n    ap.add_argument("--test", help="The test_drawN.parquet the predictions were made on. Recomputes "\n                                   "adduct_group from that draw, which is what routing actually saw.")\n    args = ap.parse_args()\n\n    sol = pd.read_csv(args.solution).set_index("molecule_id")\n    if args.test:\n        draw = pd.read_parquet(args.test, columns=["molecule_id", "adduct"])\n        sol = sol.loc[sol.index.isin(draw["molecule_id"])].copy()\n        has_base = draw.groupby("molecule_id")["adduct"].agg(\n            lambda x: any(adduct_group(a) == "base_glacier" for a in x))\n        sol["adduct_group"] = has_base.reindex(sol.index).map({True: "base_glacier", False: "ft_only"})\n    truth = sol["scorer_key"]\n    cache: dict = {}\n    rr = {}\n    for spec in args.sub:\n        name, _, path = spec.partition("=")\n        preds = load_preds(path)\n        rr[name] = reciprocal_ranks(preds, truth, cache)\n        missing = len(truth) - len(set(preds) & set(truth.index))\n        if missing:\n            print(f"warning: {name} has no predictions for {missing} molecules (scored 0)")\n    rr = pd.DataFrame(rr)\n\n    slices = {"all": pd.Series(True, index=sol.index)}\n    for col in ("adduct_group", "cls", "obscure", "timstof"):\n        if col in sol.columns and sol[col].notna().any():\n            for v in sorted(sol[col].dropna().unique(), key=str):\n                slices[f"{col}={v}"] = sol[col] == v\n\n    rows = []\n    for sname, mask in slices.items():\n        sub = rr[mask.reindex(rr.index).fillna(False)]\n        row = {"slice": sname, "n": len(sub)}\n        for name in rr.columns:\n            row[name] = sub[name].mean()\n            row[f"{name}@1"] = (sub[name] == 1).mean()\n        rows.append(row)\n    table = pd.DataFrame(rows)\n    pd.set_option("display.width", 200)\n    print(table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))\n\n    if args.baseline:\n        print(f"\\nPaired difference vs {args.baseline} (mean [90% bootstrap CI])")\n        for sname, mask in slices.items():\n            sub = rr[mask.reindex(rr.index).fillna(False)]\n            if len(sub) < 10:\n                continue\n            for name in rr.columns:\n                if name == args.baseline:\n                    continue\n                m, lo, hi = bootstrap_diff(sub[name], sub[args.baseline])\n                fixed = int(((sub[name] == 1) & (sub[args.baseline] < 1)).sum())\n                broke = int(((sub[name] < 1) & (sub[args.baseline] == 1)).sum())\n                verdict = "WIN" if lo > 0 else ("LOSS" if hi < 0 else "noise")\n                print(f"  {sname:<26} {name:<14} {m:+.4f} [{lo:+.4f}, {hi:+.4f}]  "\n                      f"top1 fixed {fixed} / broke {broke}  {verdict}")\n\n\nif __name__ == "__main__":\n    main()\n')

In [ ]:
# Downloads (all host-approved: D745148 Enveda-180, D745228 newer GNPS/MassBank, D742991 MassSpecGym)
import urllib.request, gzip, shutil
DL = '/tmp/dl'
SOURCES = {
    'enveda180': ('https://zenodo.org/api/records/21346580/files/enveda-180-filtered.parquet.gz/content', 'enveda180.parquet.gz'),
    'gnps': ('https://external.gnps2.org/gnpslibrary/ALL_GNPS_NO_PROPOGATED.mgf', 'gnps.mgf'),
    'massbank': ('https://github.com/MassBank/MassBank-data/releases/latest/download/MassBank_NISTformat.msp', 'massbank.msp'),
    'msg': ('https://huggingface.co/datasets/roman-bushuiev/MassSpecGym/resolve/main/data/MassSpecGym.tsv', 'MassSpecGym.tsv'),
}
GOT = {}
for k, (url, fn) in SOURCES.items():
    p = os.path.join(DL, fn)
    try:
        t = time.time()
        req = urllib.request.Request(url, headers={'User-Agent': 'casmi26-onp-build'})
        with urllib.request.urlopen(req, timeout=600) as r, open(p, 'wb') as f:
            shutil.copyfileobj(r, f, 1 << 24)
        if fn.endswith('.gz'):
            with gzip.open(p) as g, open(p[:-3], 'wb') as f:
                shutil.copyfileobj(g, f, 1 << 24)
            os.remove(p); p = p[:-3]
        GOT[k] = p
        print(f'{k:<10} {os.path.getsize(p)/1e9:6.2f} GB  {time.time()-t:5.0f}s')
    except Exception as e:
        print(f'{k:<10} FAILED {e!r}')
assert 'gnps' in GOT and 'msg' in GOT, 'GNPS and MassSpecGym are required'

In [ ]:
# Pool-derived helper tables: COCONUT membership, candidate-pool keys, PubChem popularity
import glob, numpy as np, pandas as pd
def find(pat):
    h = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True), key=len)
    if not h: raise FileNotFoundError(pat)
    return h[0]
pm = pd.read_parquet(find('pool_meta.parquet'))
pop = np.load(find('pool_lsid.npy')).astype(float) + np.load(find('pool_lpmid.npy')).astype(float)
assert len(pop) == len(pm), (len(pop), len(pm))
H = '/tmp/helpers'; os.makedirs(H, exist_ok=True)
# keys only (no SMILES) so load_key_set does not re-canonicalise 700k structures
pm.loc[pm.src == 2, ['key']].rename(columns={'key': 'inchikey14'}).to_parquet(f'{H}/coconut.parquet')
pm[['key']].rename(columns={'key': 'inchikey14'}).to_parquet(f'{H}/pool_keys.parquet')
pd.DataFrame({'scorer_key': pm.key, 'popularity': pop}).to_parquet(f'{H}/pool_pop.parquet')
TRAIN_STRUCTS = find('train_structs.parquet')
FT_LABELS = sorted(glob.glob('/kaggle/input/**/labels.tsv', recursive=True))
print('coconut', int((pm.src == 2).sum()), '| pool', len(pm), '| ft label files', FT_LABELS)

In [ ]:
# Build. Seen = train_structs + MassSpecGym + GLACIER fine-tune labels.
OUT = '/kaggle/working/onp'
spectra = [f'{k}={GOT[k]}' for k in ('enveda180', 'gnps', 'massbank') if k in GOT]
cmd = [sys.executable, '/kaggle/working/onp_src/build_obscure_np_set.py',
       '--train', TRAIN_STRUCTS, '--extra-seen', GOT['msg'], *FT_LABELS,
       '--spectra', *spectra,
       '--coconut', f'{H}/coconut.parquet', '--candidate-db', f'{H}/pool_keys.parquet',
       '--popularity', f'{H}/pool_pop.parquet',
       '--max-molecules', '400', '--draws', '5', '--out', OUT]
print(' '.join(cmd), flush=True)
r = subprocess.run(cmd, cwd='/kaggle/working/onp_src', capture_output=True, text=True)
print(r.stdout[-8000:]); print(r.stderr[-4000:])
assert r.returncode == 0
print(f'total {time.time()-T0:.0f}s')

In [ ]:
sol = pd.read_csv(f'{OUT}/solution.csv')
print(sol[['cls', 'obscure', 'adduct_group', 'timstof']].astype(str).value_counts().head(20))
print(sol.sources.str.split(',').explode().value_counts())
t0 = pd.read_parquet(f'{OUT}/test_draw0.parquet'); print(t0.adduct.value_counts()); print(t0.instrument_type.value_counts())